In [ ]:
%load_ext autoreload
%autoreload 2
import sys
import os
import pandas as pd
import numpy as np 
from pathlib import Path
from hypnose_analysis.utils.movement_analysis_utils import *
from hypnose_analysis.utils.visualization_utils import print_cache_keys
from ipywidgets import widgets
from IPython.display import display
%matplotlib widget

## Plot movement traces for different modes - plot_trial_trace_by_mode quick guide

- Common args: subjid, dates (list [] or range ()), xlim, ylim, smooth_window (frames), linewidth, alpha, invert_y
- Modes: 
    - rewarded: rewarded trials only
    - rewarded_hr: rewarded trials; HR trials colored with HR palette
    - completed: all completed trials (rewarded, unrewarded, timeout)
    - all_trials: completed and aborted trials
    - fa_by_response: FA trials (selected by fa_types filter), sorted by response port
    - fa_by_odor: FA trials, sorted by each aborted odor
    - hr_only: hidden-rule trials, colored by associated reward port, with rewarded and unrewarded trials
- Options: 
    - show_average: adds mean trace + SEM per category
    - highlight_hr: in rewarded/all_trials mode, recolor HR trials in different palette
    - color_by_index: ignore categories; color each trace by normalized sample index
    - fa_types: filter FA labels (select between "FA_time_in", "FA_time_out", or both)

In [ ]:
plot_trial_traces_by_mode(
    subjid=40,
    dates=[20251217],
    mode='rewarded',
    xlim=(100,1160),
    ylim=(10,950),
    show_average=False, 
    highlight_hr=True, 
    color_by_index=False,
    color_by_speed=False,
    color_by_trial_id=False,
    fa_types=['FA_time_in'],
    figsize=(10,6),
    save=True
)

# Modes: 
    # rewarded, rewarded_hr, completed, all_trials, fa_by_response, fa_by_odor, hr_only

# Speed Analysis

In [ ]:
speed_analysis = run_speed_analysis_batch(
    subjids=[45],
    dates=None,
    fa_label_filter=["fa_time_in", "fa_time_out"],
    threshold=True
)

In [ ]:
speed_analysis_plot = plot_epoch_speeds_by_condition(
    subjid=45,
    dates=[20260205],
    fa_label_filter=["fa_time_in", "fa_time_out"],
    save=True
)


In [ ]:
fig_thresh = plot_traces_with_speed_threshold(
    subjid=45,
    dates=[20260219],
    fa_types=["FA_time_in"],
    pre_buffer_s=0.5,
    threshold_alpha=10.0,
    threshold_beta=10.0,
    smooth_window=5,
    invert_y=True, 
    save=True
)

In [ ]:
figs_overlay = plot_tortuosity_lines_overlay(
    subjid=40,
    dates=[20251223],
    fa_types=["FA_time_in"],
    bin_ms=100,
    fixed_start_xy=(575, 90),
    fixed_goal_a_xy=(208, 930),
    fixed_goal_b_xy=(973, 930),
    save=True
)

In [ ]:
onset_latency = plot_movement_analysis_statistics(
    subjid=40,
    dates=[20251223],
    fa_types=["FA_time_in"],
    clean_graph=False,
    hidden_rule_analysis=True,
    save=True
)

# Functionalities

In [ ]:
plt.close('all')

In [ ]:
print_cache_keys()

# Miscellaneous

In [ ]:
# modes can be simple (all movement), trial_state (within trial vs outside), last_odor (A vs B), trial_windows (one or more trial windows), time_windows (one or more time windows), or trial_windows_rew
# for trial_windows: trial_windows=[(0, 20), (-20, None)] will plot first vs last 20 trials
# for time_windows: time_windows=[("15:20:00","15:25:00"), ("16:00:00","16:05:00")] will plot 2 5-minute windows
plot_movement_with_behavior(40, 20251211, mode='time_windows', time_windows=[("14:42:12", "14:42:55")],trial_windows=[(0, 10), (-10,None)], xlim=(100,1160), ylim=(10,950))

In [ ]:
# Cell to check speed bins for high speeds in a time window before 0. This is for rewarded trials only. Returns trial ID of trials that have speed above threshold in the pre-zero window.

from pathlib import Path
import pandas as pd
from hypnose_analysis.paths import get_derivatives_root

# Scan speed bins for high speeds in the pre-zero window (rewarded trials only)
subjid = 45
date = 20260205
speed_threshold = 200.0
t_min, t_max = -0.4, -0.001  # seconds relative to t=0

sub_str = f"sub-{subjid:03d}"
deriv_root = get_derivatives_root()
sub_dirs = list(deriv_root.glob(f"{sub_str}_id-*"))
if not sub_dirs:
    raise FileNotFoundError(f"No subject dir for {sub_str} under {deriv_root}")
sub_dir = sub_dirs[0]

ses_dirs = list(sub_dir.glob(f"ses-*_date-{date}"))
if not ses_dirs:
    raise FileNotFoundError(f"No session dir for date {date}")
ses_dir = ses_dirs[0]

results_dir = ses_dir / "saved_analysis_results"
analysis_path = results_dir / "speed_analysis.parquet"
trial_path_parquet = results_dir / "trial_data.parquet"
trial_path_csv = results_dir / "trial_data.csv"

if not analysis_path.exists():
    raise FileNotFoundError(f"speed_analysis.parquet not found at {analysis_path}")

df = pd.read_parquet(analysis_path)

# Optional: load trial metadata to report trial_id if available
trial_df = None
if trial_path_parquet.exists():
    trial_df = pd.read_parquet(trial_path_parquet)
elif trial_path_csv.exists():
    trial_df = pd.read_csv(trial_path_csv)

# Filter to rewarded trials
if "condition" in df.columns:
    df = df[df["condition"] == "rewarded"].copy()
rewarded_indices = None
if trial_df is not None and "response_time_category" in trial_df.columns:
    rewarded_mask = (trial_df["response_time_category"].str.lower() == "rewarded") & (~trial_df.get("is_aborted", False).astype(bool))
    rewarded_indices = set(trial_df.index[rewarded_mask].tolist())
    df = df[df["trial_index"].isin(rewarded_indices)]

# Apply window and speed filters
mask_window = df["bin_mid_s"].between(t_min, t_max, inclusive="both")
mask_speed = df["speed"] > speed_threshold
suspect = df[mask_window & mask_speed].copy()

if suspect.empty:
    print("No bins exceed threshold in the requested window (rewarded trials only).")
else:
    trial_counts = suspect.groupby("trial_index").size().sort_index()
    print(f"Found {len(suspect)} bins across {trial_counts.size} rewarded trials")
    print("Trial indices:", list(trial_counts.index))
    if trial_df is not None and "trial_id" in trial_df.columns:
        id_map = trial_df["trial_id"]
        labeled = [id_map.get(idx, idx) if hasattr(id_map, "get") else idx for idx in trial_counts.index]
        print("Trial IDs:", labeled)
    display(suspect.sort_values(["trial_index", "bin_mid_s"])[["trial_index", "bin_mid_s", "speed"]].head(30))